In [1]:
!pip install corus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.8 MB/s eta 0:00:00


In [2]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 22.3 MB/s eta 0:00:00


In [3]:
import corus
import pandas as pd
import os
import requests
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter

In [4]:
# URL датасета
LENTA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
LOCAL_FILE = "lenta-ru-news.csv.gz"

random_state=42

def download_file(url, path):
    if not os.path.exists(path):
        print("Файл не найден, скачивание...")
        response = requests.get(url, stream=True)
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024):
                file.write(chunk)
        print("Загрузка завершена.")

def load_lenta_dataset(path=LOCAL_FILE, sample_size=100000):
    download_file(LENTA_URL, path)  # Проверка и скачивание файла
    records = []
    dataset = corus.load_lenta(path)

    for record in tqdm(dataset, desc="Загрузка записей"):
        records.append((record.title, record.text, record.topic))

    df = pd.DataFrame(records, columns=["title", "text", "topic"])

    df = df.sample(n=sample_size, random_state=42)

    topic_counts = df["topic"].value_counts()
    valid_topics = topic_counts[topic_counts >= 500].index
    df = df[df["topic"].isin(valid_topics)]

    min_class_size = min(Counter(df["topic"]).values())
    df_balanced = df.groupby("topic").apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)

    return df_balanced

data = load_lenta_dataset()

data.head()


Файл не найден, скачивание...
Загрузка завершена.


Загрузка записей: 739351it [00:43, 16931.44it/s]
<ipython-input-4-a5fc213cfade>:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("topic").apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)


,title,text,topic
0,Газета сообщила о грядущем возвращении Кириенк...,Первый замглавы администрации президента Серге...,Бизнес
1,В «Аэрофлоте» назвали число перевезенных пасса...,"Группа «Аэрофлот» перевезла 1,925 миллиона (94...",Бизнес
2,Роспотребнадзор разрешил поставки 20 видов мол...,Федеральная служба по надзору в сфере защиты п...,Бизнес
3,Бывший владелец Черкизовского рынка подал заяв...,"Бывший владелец Черкизовского рынка, основател...",Бизнес
4,Суд Дюссельдорфа снял обеспечительные меры на ...,Высший земельный суд Дюссельдорфа снял обеспеч...,Бизнес


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11970 entries, 0 to 11969
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   11970 non-null  object
 1   text    11970 non-null  object
 2   topic   11970 non-null  object
dtypes: object(3)
memory usage: 280.7+ KB


In [ ]:
data.topic.value_counts()

,count
topic,
Бизнес,855
Бывший СССР,855
Дом,855
Из жизни,855
Интернет и СМИ,855
Культура,855
Мир,855
Наука и техника,855
Путешествия,855


Для банлансировки классов устанавливаем порог в 500 значений, что позволила нам получить 14 классов

In [6]:

import re
import pymorphy3
import pandas as pd

morph = pymorphy3.MorphAnalyzer()

def clean_text(text):
    text = re.sub(r'[^А-Яа-яЁё\s]', '', text)
    text = re.sub(r'\s+', ' ', text)  # Удаление лишних пробелов
    text = text.strip().lower()  # Приведение к нижнему регистру
    return text

def lemmatize_text(text):
    words = text.split()  # Разбиваем текст на отдельные слова
    lemmatized_words = []

    for word in words:
        parsed_word = morph.parse(word)[0]  # Получаем анализатор для слова
        lemmatized_words.append(parsed_word.normal_form)  # Получаем лемму

    return ' '.join(lemmatized_words)

def normalize_text(text):
    # Очистка текста
    cleaned_text = clean_text(text)
    # Лемматизация текста
    lemmatized_text = lemmatize_text(cleaned_text)
    return lemmatized_text

Для дальнейшей обработки текста будем использовать пакет pymorphy

In [7]:
data['cleaned_title'] = data['title'].apply(normalize_text)
data['cleaned_text'] = data['text'].apply(normalize_text)
data['cleaned_topic'] = data['topic'].apply(normalize_text)

In [8]:
df = data.drop(columns=['title', 'text', 'topic'])

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder


train_df, temp_df = train_test_split(df, test_size=0.4, stratify=df['cleaned_topic'], random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['cleaned_topic'], random_state=42)

vectorizer = TfidfVectorizer()
label_encoder = LabelEncoder()

X_train_title = vectorizer.fit_transform(train_df['cleaned_title'])
X_train_text = vectorizer.fit_transform(train_df['cleaned_text'])

y_train = label_encoder.fit_transform(train_df[['cleaned_topic']])

X_test_title = vectorizer.transform(test_df['cleaned_title'])
X_test_text = vectorizer.transform(test_df['cleaned_text'])

y_test = label_encoder.transform(test_df[['cleaned_topic']])

X_train = pd.concat([pd.DataFrame(X_train_title.toarray()), pd.DataFrame(X_train_text.toarray())], axis=1)
X_test = pd.concat([pd.DataFrame(X_test_title.toarray()), pd.DataFrame(X_test_text.toarray())], axis=1)

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [ ]:
model = DummyClassifier(strategy='most_frequent', random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Базовое качество (accuracy) с использованием DummyClassifier: {accuracy:.4f}")

Базовое качество (accuracy) с использованием DummyClassifier: 0.0714


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder


train_df, temp_df = train_test_split(df, test_size=0.4, stratify=df['cleaned_topic'], random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['cleaned_topic'], random_state=42)

label_encoder = LabelEncoder()
train_df['encoded_topic'] = label_encoder.fit_transform(train_df['cleaned_topic'])
test_df['encoded_topic'] = label_encoder.transform(test_df['cleaned_topic'])

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(train_df['cleaned_title'] + ' ' + train_df['cleaned_text'])
X_test_count = count_vectorizer.transform(test_df['cleaned_title'] + ' ' + test_df['cleaned_text'])

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['cleaned_title'] + ' ' + train_df['cleaned_text'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['cleaned_title'] + ' ' + test_df['cleaned_text'])

logreg_count = LogisticRegression(max_iter=1000, random_state=42)
logreg_count.fit(X_train_count, train_df['encoded_topic'])

predictions_count = logreg_count.predict(X_test_count)

accuracy_count = accuracy_score(test_df['encoded_topic'], predictions_count)
print(f"Точность с CountVectorizer: {accuracy_count:.4f}")

logreg_tfidf = LogisticRegression(max_iter=1000, random_state=42)
logreg_tfidf.fit(X_train_tfidf, train_df['encoded_topic'])

predictions_tfidf = logreg_tfidf.predict(X_test_tfidf)

accuracy_tfidf = accuracy_score(test_df['encoded_topic'], predictions_tfidf)
print(f"Точность с TfidfVectorizer: {accuracy_tfidf:.4f}")


Точность с CountVectorizer: 0.7749
Точность с TfidfVectorizer: 0.7786


Данные векторизированные с помощью Tfid дают большую точность

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

df = df.dropna(subset=['cleaned_title', 'cleaned_text', 'cleaned_topic'])

train_df, temp_df = train_test_split(df, test_size=0.4, stratify=df['cleaned_topic'], random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['cleaned_topic'], random_state=42)

train_texts = train_df['cleaned_title'] + " " + train_df['cleaned_text']
test_texts = test_df['cleaned_title'] + " " + test_df['cleaned_text']

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['cleaned_topic'])
y_test = label_encoder.transform(test_df['cleaned_topic'])

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

param_grid = {
    'vectorizer__max_features': [5000, 10000, None],
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'classifier__C': [0.1, 1, 10]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(train_texts, y_train)
print(f"Лучшие параметры: {grid_search.best_params_}")

predictions = grid_search.predict(test_texts)

accuracy = accuracy_score(y_test, predictions)
print(f"Оптимальное качество (accuracy): {accuracy:.4f}")



Fitting 5 folds for each of 18 candidates, totalling 90 fits
Лучшие параметры: {'classifier__C': 10, 'vectorizer__max_features': None, 'vectorizer__ngram_range': (1, 2)}
Оптимальное качество (accuracy): 0.7941
